# SkySim — GPU-first PPO (CUDA)

این مسیر TPU ندارد و برای GPU انویدیا ساخته شده است. قبل از آموزش طولانی، سلول تست را اجرا کن.

**محدودیت علمی مهم:** این نسخه، فیزیک برداری‌شدهٔ آزمایشی است؛ هنوز هم‌ارزی با DroneCore C++ ثابت نشده و مدل کامل ملخ/آیرودینامیک، ground-effect/VRS و باتری عیناً منتقل نشده‌اند. بنابراین این PR فعلاً جایگزین تأییدشدهٔ آموزش اصلی نیست.

In [ ]:
import torch, os, subprocess, sys
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('در Runtime گزینهٔ GPU را انتخاب کن؛ اجرای CPU مجاز نیست.')
print('GPU:', torch.cuda.get_device_name(0))
repo='/content/3d-fly-test'
if not os.path.isdir(repo): subprocess.check_call(['git','clone','--branch','feat/cuda-vectorized-physics-ppo','https://github.com/samanramezani1377-hub/3d-fly-test.git',repo])
%cd /content/3d-fly-test
subprocess.check_call([sys.executable, 'training/test_cuda_path.py'])

## اجرای آموزش یک مدل مشترک

تنظیمات درخواستی حفظ شده‌اند: **۴۰ محیط، ۱۰۰۰ گام rollout برای هر محیط، batch=4000 و مجموع ۴۰٬۰۰۰٬۰۰۰ انتقال محیطی**. checkpoint هر ۱۰۰٬۰۰۰ انتقال ذخیره می‌شود. این اجرا می‌تواند طولانی باشد و فقط پس از موفقیت تست CUDA اجرا شود.

In [ ]:
!python training/train_ppo_cuda.py --timesteps 40000000 --n-envs 40 --n-steps 1000 --batch-size 4000 --save-freq 100000 --run-name cuda-ppo-40m

## محل خروجی‌ها

مدل نهایی و checkpointها در `training/runs/cuda-ppo-40m/` قرار می‌گیرند. برای قطع و ادامهٔ آموزش، باید منطق resume از checkpoint به runner اضافه شود؛ این نسخه فعلاً checkpoint ذخیره می‌کند اما resume خودکار ندارد. قبل از استفاده از مدل، ابتدا فیزیک CUDA باید با DroneCore مرجع تطبیق داده شود.